In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

In [ ]:
from pandas.core.frame import DataFrame


df: DataFrame = pd.read_csv(
    "../data/sim_reqs.csv", parse_dates=["req_hour"], index_col="req_hour"
)

In [21]:
# Sort by time and visualize request count over time
df_sorted = df.sort_index()

fig = px.line(
    df_sorted.reset_index(),
    x="req_hour",
    y="reqs",
    title="Request Count Over Time",
    labels={"req_hour": "Time", "reqs": "Request Count"},
)
fig.update_layout(template="plotly_white")
fig.show()

In [26]:
# Q-Q plot to test if data is log-normal
# If log-normal: log(data) should follow a normal distribution

from scipy import stats

# Log-transform the data (filter out zeros first)
log_reqs = np.log(df["reqs"][df["reqs"] > 0])

# Calculate theoretical quantiles
sorted_log_data = np.sort(log_reqs)
n = len(sorted_log_data)
# PPF = Percent Point Function (also called the Inverse Cumulative Distribution Function or Quantile Function)
# It answers: "At what value X does the cumulative probability equal p?"
# CDF: "What's the probability of being ≤ 1.96?"
# norm.cdf(1.96)  # → 0.975

# PPF: "What value has 97.5% of data below it?"
# norm.ppf(0.975)  # → 1.96
theoretical_quantiles = stats.norm.ppf(np.arange(1, n + 1) / (n + 1))

# Create Q-Q plot with Plotly
fig = px.scatter(
    x=theoretical_quantiles,
    y=sorted_log_data,
    labels={
        "x": "Theoretical Quantiles (Normal)",
        "y": "Sample Quantiles (Log of Requests)",
    },
    title="Q-Q Plot: Log(Requests) vs Normal Distribution",
)

# Add reference line (if log-normal, points should follow this line)
slope, intercept = np.polyfit(theoretical_quantiles, sorted_log_data, 1)
x_line = np.array([theoretical_quantiles.min(), theoretical_quantiles.max()])
y_line = slope * x_line + intercept
fig.add_scatter(
    x=x_line,
    y=y_line,
    mode="lines",
    name="Reference Line",
    line=dict(color="red", dash="dash"),
)

fig.update_layout(template="plotly_white")
fig.show()

# Print interpretation
print(f"If points follow the red line, data is log-normal.")
print(f"Log-transformed mean: {log_reqs.mean():.2f}, std: {log_reqs.std():.2f}")

If points follow the red line, data is log-normal.
Log-transformed mean: 1.68, std: 1.27


In [27]:
# Log-Log plot to check distribution type
# Power-law: straight line on log-log plot
# Poisson: curved on log-log plot

# Get the distribution of request counts
# Count frequency of each request value
size_counts = df["reqs"].value_counts().reset_index()
size_counts.columns = ["reqs", "frequency"]
size_counts = size_counts.sort_values("reqs")

# Filter out zeros/negatives for log scale
size_counts = size_counts[size_counts["reqs"] > 0]
size_counts = size_counts[size_counts["frequency"] > 0]

# Create log-log plot
fig = px.scatter(
    size_counts,
    x="reqs",
    y="frequency",
    log_x=True,
    log_y=True,
    title="Log-Log Plot: Request Count Distribution",
    labels={"reqs": "Request Count", "frequency": "Frequency"},
)

fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.update_layout(
    xaxis_title="Request Count (log scale)",
    yaxis_title="Frequency (log scale)",
    template="plotly_white",
)
fig.show()

In [ ]:
# Alternative: Use logarithmic bins for cleaner visualization
# This groups similar request counts together

req_counts = df["reqs"].dropna()
req_counts = req_counts[req_counts > 0]

# Create logarithmic bins
log_bins = np.logspace(np.log10(req_counts.min()), np.log10(req_counts.max()), 30)
hist, bin_edges = np.histogram(req_counts, bins=log_bins)

# Use bin centers for x-axis
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean

# Create dataframe for plotting
hist_df = pd.DataFrame({"reqs": bin_centers, "frequency": hist})
hist_df = hist_df[hist_df["frequency"] > 0]

fig2 = px.scatter(
    hist_df,
    x="reqs",
    y="frequency",
    log_x=True,
    log_y=True,
    title="Log-Log Plot (Binned): Request Count Distribution",
    labels={"reqs": "Request Count", "frequency": "Frequency"},
)

fig2.update_traces(marker=dict(size=10, opacity=0.8))
fig2.update_layout(
    xaxis_title="Request Count (log scale)",
    yaxis_title="Frequency (log scale)",
    template="plotly_white",
)

# Add interpretation note
print("Interpretation:")
print("- Straight line → Power-law distribution")
print("- Curved (concave down) → Poisson/Exponential distribution")
print("- Curved (concave up) → Log-normal or similar")

fig2.show()

Interpretation:
- Straight line → Power-law distribution
- Curved (concave down) → Poisson/Exponential distribution
- Curved (concave up) → Log-normal or similar
